# Avito: итоговое решение

Ноутбук воспроизводит шестую посылку (`Recall@50 = 0.799730`) и
создаёт `answer.csv`. Параметры зафиксированы по результатам отдельных
экспериментов; валидация и перебор конфигураций здесь не выполняются.

Итоговая выдача объединяет BM25F, историю взаимодействий, предсказанную
микрокатегорию, векторный поиск E5 и перенос локации по истории.


## Окружение

Дополнительные библиотеки зафиксированы на версиях успешного запуска.
PyTorch и CUDA используются из образа Kaggle.


In [ ]:
%pip install -q pymorphy3==2.0.6 sentence-transformers==6.1.0 transformers==5.17.0 faiss-cpu==1.15.1


In [ ]:
from collections import defaultdict
from functools import lru_cache
import gc
import hashlib
import platform
import random
import re
import time
import unicodedata

import faiss
import numpy as np
import pandas as pd
import pymorphy3
import scipy
import sklearn
import sentence_transformers
import torch
import transformers
from scipy.sparse import csr_matrix
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import FeatureUnion

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}; pandas: {pd.__version__}; SciPy: {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}; PyTorch: {torch.__version__}")
print(
    f"sentence-transformers: {sentence_transformers.__version__}; "
    f"transformers: {transformers.__version__}; faiss: {faiss.__version__}"
)
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
from pathlib import Path
import os

# Для локального запуска достаточно изменить PROJECT_DIR или задать
# AVITO_PROJECT_DIR / AVITO_DATA_DIR / AVITO_OUTPUT_DIR.
KAGGLE_DATA_DIR = Path("/kaggle/input/datasets/yuriymagus/avito-nlp-interns")
DEFAULT_PROJECT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

PROJECT_DIR = Path(os.environ.get("AVITO_PROJECT_DIR", DEFAULT_PROJECT_DIR)).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get(
        "AVITO_DATA_DIR",
        KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else PROJECT_DIR / "data",
    )
).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("AVITO_OUTPUT_DIR", PROJECT_DIR)).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = (
    "train.parquet",
    "benchmark_queries.parquet",
    "benchmark_items.parquet",
)
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        f"Не найдены файлы {missing_files} в {DATA_DIR}. "
        "Измените DATA_DIR в первой ячейке."
    )

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"OUTPUT_DIR:  {OUTPUT_DIR}")


In [ ]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print(f"train:   {train.shape}")
print(f"queries: {queries.shape}")
print(f"items:   {items.shape}")


## 1. BM25F по заголовку и описанию

Текст приводится к нижнему регистру, `ё` заменяется на `е`, русские
слова лемматизируются. Заголовок имеет вес `1`, описание — `1/3`.
При одинаковой оценке объявления упорядочиваются по `item_id`.


In [ ]:
TOKEN_PATTERN = re.compile(r"[0-9a-zа-я]+")
CYRILLIC_TOKEN_PATTERN = re.compile(r"^[а-я]+$")
morph = pymorphy3.MorphAnalyzer()


def normalize_lexical_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.casefold().replace("ё", "е")
    return " ".join(TOKEN_PATTERN.findall(text))


@lru_cache(maxsize=500_000)
def lemmatize_token(token: str) -> str:
    if not CYRILLIC_TOKEN_PATTERN.fullmatch(token):
        return token
    return morph.parse(token)[0].normal_form.replace("ё", "е")


def lemmatize_text(value: object) -> str:
    normalized = normalize_lexical_text(value)
    return " ".join(lemmatize_token(token) for token in normalized.split())


def normalize_identifier(values: pd.Series) -> np.ndarray:
    return (
        pd.to_numeric(values, errors="coerce")
        .astype("Int64")
        .astype("string")
        .fillna("<NA>")
        .to_numpy()
    )


def make_item_tie_keys(item_ids: np.ndarray) -> np.ndarray:
    '''Целочисленный ключ сохраняет лексикографический порядок item_id.'''
    item_ids = np.asarray(item_ids, dtype=str)
    order = np.argsort(item_ids, kind="mergesort")
    keys = np.empty(len(item_ids), dtype=np.int64)
    keys[order] = np.arange(len(item_ids), dtype=np.int64)
    return keys


In [ ]:
def idf_from_document_frequency(
    document_frequency: np.ndarray,
    n_documents: int,
) -> np.ndarray:
    return np.log1p(
        (n_documents - document_frequency + 0.5)
        / (document_frequency + 0.5)
    ).astype(np.float32)


def normalize_field_term_frequency(
    count_matrix: csr_matrix,
    b: float = 0.75,
) -> csr_matrix:
    '''Нормализует поле на независимой копии и не меняет count_matrix.'''
    matrix = count_matrix.tocsr(copy=True).astype(np.float32, copy=False)
    field_lengths = np.asarray(matrix.sum(axis=1)).ravel().astype(np.float32)
    average_field_length = float(field_lengths.mean())

    if average_field_length == 0.0:
        return matrix

    row_normalization = 1.0 / (
        1.0 - b + b * field_lengths / average_field_length
    )
    matrix.data *= np.repeat(row_normalization, np.diff(matrix.indptr))
    return matrix


def build_bm25f_matrix(
    title_term_frequency: csr_matrix,
    description_term_frequency: csr_matrix,
    inverse_document_frequency: np.ndarray,
    title_weight: float = 1.0,
    description_weight: float = 1.0 / 3.0,
    k1: float = 1.2,
) -> csr_matrix:
    '''B8: веса title=1, description=1/3.'''
    combined_tf = (
        title_term_frequency.multiply(title_weight)
        + description_term_frequency.multiply(description_weight)
    ).tocsr()
    combined_tf.sum_duplicates()
    combined_tf.eliminate_zeros()

    combined_tf.data = (
        combined_tf.data * (k1 + 1.0)
        / (combined_tf.data + k1)
    )
    combined_tf.data *= inverse_document_frequency[combined_tf.indices]
    return combined_tf


In [ ]:
def select_exact_top_k(
    indices: np.ndarray,
    values: np.ndarray,
    item_tie_keys: np.ndarray,
    k: int,
    mask: np.ndarray | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    '''Детерминированный top-k с учётом равных значений на границе.'''
    if mask is not None:
        indices = indices[mask]
        values = values[mask]

    if len(indices) == 0:
        return (
            np.empty(0, dtype=np.int32),
            np.empty(0, dtype=np.float32),
        )

    order = np.lexsort((item_tie_keys[indices], -values))
    selected = order[: min(k, len(order))]
    return (
        indices[selected].astype(np.int32, copy=False),
        values[selected].astype(np.float32, copy=False),
    )


def retrieve_global_and_location(
    query_vectors: csr_matrix,
    item_vectors: csr_matrix,
    query_locations: np.ndarray,
    item_locations: np.ndarray,
    item_tie_keys: np.ndarray,
    global_k: int = 200,
    location_k: int = 100,
    batch_size: int = 16,
) -> dict[str, list[np.ndarray] | np.ndarray]:
    '''За один расчёт строит глобальную выдачу и выдачу по локации.'''
    result = {
        "global_indices": [],
        "global_scores": [],
        "location_indices": [],
        "location_scores": [],
        "nonzero_counts": np.zeros(query_vectors.shape[0], dtype=np.int32),
    }

    for batch_start in range(0, query_vectors.shape[0], batch_size):
        batch_end = min(batch_start + batch_size, query_vectors.shape[0])
        scores = (query_vectors[batch_start:batch_end] @ item_vectors.T).tocsr()

        for local_row in range(scores.shape[0]):
            query_position = batch_start + local_row
            row_start = scores.indptr[local_row]
            row_end = scores.indptr[local_row + 1]
            indices = scores.indices[row_start:row_end]
            values = scores.data[row_start:row_end]
            result["nonzero_counts"][query_position] = len(indices)

            global_indices, global_scores = select_exact_top_k(
                indices, values, item_tie_keys, global_k
            )
            location_mask = (
                item_locations[indices] == query_locations[query_position]
            )
            location_indices, location_scores = select_exact_top_k(
                indices,
                values,
                item_tie_keys,
                location_k,
                mask=location_mask,
            )

            result["global_indices"].append(global_indices)
            result["global_scores"].append(global_scores)
            result["location_indices"].append(location_indices)
            result["location_scores"].append(location_scores)

    return result


In [ ]:
def build_b8_index(item_frame: pd.DataFrame) -> dict:
    start = time.perf_counter()

    title_texts = [lemmatize_text(value) for value in item_frame["item_title_raw"]]
    description_texts = [
        lemmatize_text(value) for value in item_frame["item_description_raw"]
    ]

    vectorizer = CountVectorizer(
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 1),
        dtype=np.float32,
    )

    combined_counts = vectorizer.fit_transform(
        f"{title} {description}"
        for title, description in zip(title_texts, description_texts)
    ).tocsr()

    n_documents, n_terms = combined_counts.shape
    document_frequency = np.bincount(
        combined_counts.indices,
        minlength=n_terms,
    ).astype(np.float32)
    inverse_document_frequency = idf_from_document_frequency(
        document_frequency,
        n_documents,
    )

    title_counts = vectorizer.transform(title_texts).tocsr()
    description_counts = vectorizer.transform(description_texts).tocsr()

    title_before = title_counts.data.copy()
    description_before = description_counts.data.copy()

    title_tf = normalize_field_term_frequency(title_counts, b=0.75)
    description_tf = normalize_field_term_frequency(description_counts, b=0.75)

    assert np.array_equal(title_counts.data, title_before)
    assert np.array_equal(description_counts.data, description_before)

    item_matrix = build_bm25f_matrix(
        title_term_frequency=title_tf,
        description_term_frequency=description_tf,
        inverse_document_frequency=inverse_document_frequency,
        title_weight=1.0,
        description_weight=1.0 / 3.0,
        k1=1.2,
    )

    item_ids = item_frame["item_id"].astype("string").to_numpy()
    item_locations = normalize_identifier(item_frame["item_location_id"])
    item_tie_keys = make_item_tie_keys(item_ids)

    del combined_counts, title_counts, description_counts, title_tf, description_tf
    gc.collect()

    print(
        f"B8 index: {item_matrix.shape}, nnz={item_matrix.nnz:,}, "
        f"время={time.perf_counter() - start:.1f} с"
    )

    return {
        "vectorizer": vectorizer,
        "item_matrix": item_matrix,
        "item_ids": item_ids,
        "item_locations": item_locations,
        "item_tie_keys": item_tie_keys,
    }


def transform_queries(
    query_frame: pd.DataFrame,
    vectorizer: CountVectorizer,
) -> tuple[list[str], csr_matrix]:
    lemma_queries = [lemmatize_text(value) for value in query_frame["search_query"]]
    query_counts = vectorizer.transform(lemma_queries).tocsr()
    query_counts.data.fill(1.0)
    return lemma_queries, query_counts


In [ ]:
def calculate_hits(
    rankings: list[np.ndarray],
    item_ids: np.ndarray,
    target_ids: np.ndarray,
    k: int = 50,
) -> np.ndarray:
    return np.fromiter(
        (
            target_id in item_ids[ranking[:k]]
            for target_id, ranking in zip(target_ids, rankings)
        ),
        dtype=bool,
        count=len(target_ids),
    )


def build_fallback_orders(
    item_frame: pd.DataFrame,
    interaction_events: pd.DataFrame,
) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    '''Резервный порядок: история → рейтинг → отзывы → item_id.'''
    item_ids = item_frame["item_id"].astype("string")
    history = (
        interaction_events["item_id"].astype("string").value_counts()
    )
    frame = pd.DataFrame({
        "item_index": np.arange(len(item_frame), dtype=np.int32),
        "item_id": item_ids.to_numpy(),
        "history": item_ids.map(history).fillna(0).to_numpy(),
        "rating": pd.to_numeric(item_frame["item_rating"], errors="coerce")
        .fillna(-1).to_numpy(),
        "reviews": pd.to_numeric(
            item_frame["item_rating_reviews_count"], errors="coerce"
        ).fillna(-1).to_numpy(),
        "location": normalize_identifier(item_frame["item_location_id"]),
    })
    ordered = frame.sort_values(
        ["history", "rating", "reviews", "item_id"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    global_order = ordered["item_index"].to_numpy(dtype=np.int32)
    assert len(global_order) == len(item_frame)
    assert np.unique(global_order).size == len(item_frame)
    by_location = {
        str(location): group["item_index"].to_numpy(dtype=np.int32)
        for location, group in ordered.groupby("location", sort=False)
    }
    return global_order, by_location


def fuse_local_global_top50(
    global_rankings: list[np.ndarray],
    location_rankings: list[np.ndarray],
    query_locations: np.ndarray,
    global_fallback: np.ndarray,
    location_fallback: dict[str, np.ndarray],
    local_quota: int,
    k: int = 50,
) -> list[np.ndarray]:
    '''Берёт квоту по локации, затем глобальную выдачу и резерв.'''
    final_rankings = []

    global_fallback = np.asarray(global_fallback, dtype=np.int32).ravel()

    for query_position, (global_ranking, local_ranking, query_location) in enumerate(zip(
        global_rankings,
        location_rankings,
        query_locations,
    )):
        selected = []
        seen = set()

        def add_until(indices, max_total):
            for raw_item_index in indices:
                if len(selected) >= max_total:
                    break

                item_index = int(raw_item_index)
                if item_index not in seen:
                    seen.add(item_index)
                    selected.append(item_index)

        add_until(local_ranking, max_total=min(local_quota, k))
        add_until(global_ranking, max_total=k)

        if len(selected) < k:
            add_until(location_fallback.get(str(query_location), ()), max_total=k)
        if len(selected) < k:
            add_until(global_fallback, max_total=k)

        if len(selected) != k:
            raise RuntimeError(
                f"Не удалось заполнить top-{k} для query_position={query_position}: "
                f"selected={len(selected)}, global_nonzero={len(global_ranking)}, "
                f"location_nonzero={len(local_ranking)}, "
                f"global_fallback={len(global_fallback)}, "
                f"location={query_location!r}"
            )

        assert len(set(selected)) == k
        final_rankings.append(np.asarray(selected, dtype=np.int32))

    return final_rankings


In [ ]:
GLOBAL_DEPTH = 1_000
LOCATION_DEPTH = 500
BASE_LOCATION_QUOTA = 50

lexical_index = build_b8_index(items)
lemma_queries, query_counts = transform_queries(
    queries,
    lexical_index["vectorizer"],
)
query_locations = normalize_identifier(queries["search_location_id"])

start = time.perf_counter()
lexical_retrieval = retrieve_global_and_location(
    query_vectors=query_counts,
    item_vectors=lexical_index["item_matrix"],
    query_locations=query_locations,
    item_locations=lexical_index["item_locations"],
    item_tie_keys=lexical_index["item_tie_keys"],
    global_k=GLOBAL_DEPTH,
    location_k=LOCATION_DEPTH,
    batch_size=16,
)
print(f"BM25F-поиск: {time.perf_counter() - start:.1f} с")

global_fallback, location_fallback = build_fallback_orders(items, train)
lexical_top50 = fuse_local_global_top50(
    global_rankings=lexical_retrieval["global_indices"],
    location_rankings=lexical_retrieval["location_indices"],
    query_locations=query_locations,
    global_fallback=global_fallback,
    location_fallback=location_fallback,
    local_quota=BASE_LOCATION_QUOTA,
    k=50,
)
print(f"Сформировано выдач: {len(lexical_top50):,}")


## 2. Объявления из истории одинаковых запросов

Для повторяющихся формулировок сначала используются объявления из
train, которые присутствуют в benchmark-корпусе. Ключи проверяются от
полного контекста запроса к одной нормализованной формулировке.


In [ ]:
def normalize_query_group(values: pd.Series) -> pd.Series:
    """Нормализация формулировки для поиска одинаковых запросов."""
    return (
        values.astype("string")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


In [ ]:
def normalize_params_text(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .fillna("")
        .str.casefold()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def make_history_key_frame(frame: pd.DataFrame) -> pd.DataFrame:
    result = pd.DataFrame({
        "__query": normalize_query_group(frame["search_query"]).to_numpy(),
        "__location": normalize_identifier(frame["search_location_id"]),
        "__category": normalize_identifier(frame["search_category"]),
        "__delivery": pd.to_numeric(
            frame["search_is_delivery_search"], errors="coerce"
        ).fillna(-1).astype(np.int16).to_numpy(),
        "__params": normalize_params_text(
            frame["search_infm_params_text"]
        ).to_numpy(),
    })

    if "item_id" in frame.columns:
        result["__item_id"] = frame["item_id"].astype("string").to_numpy()

    return result


def build_ranked_history_map(
    history_frame: pd.DataFrame,
    key_columns: list[str],
) -> dict[tuple, list[str]]:
    counts = (
        history_frame
        .groupby(key_columns + ["__item_id"], dropna=False)
        .size()
        .rename("__count")
        .reset_index()
        .sort_values(
            key_columns + ["__count", "__item_id"],
            ascending=[True] * len(key_columns) + [False, True],
            kind="mergesort",
        )
    )

    result = {}
    grouper = key_columns[0] if len(key_columns) == 1 else key_columns

    for key, group in counts.groupby(grouper, sort=False, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        result[key] = group["__item_id"].astype(str).tolist()

    return result


def build_history_maps(
    history_events: pd.DataFrame,
    allowed_item_ids: set[str],
) -> dict[str, dict[tuple, list[str]]]:
    history_frame = make_history_key_frame(history_events)
    history_frame = history_frame.loc[
        history_frame["__item_id"].isin(allowed_item_ids)
    ].copy()

    return {
        "context": build_ranked_history_map(
            history_frame,
            ["__query", "__location", "__category", "__delivery", "__params"],
        ),
        "query_location": build_ranked_history_map(
            history_frame,
            ["__query", "__location"],
        ),
        "query": build_ranked_history_map(
            history_frame,
            ["__query"],
        ),
    }


def get_history_candidates(
    query_frame: pd.DataFrame,
    history_maps: dict[str, dict[tuple, list[str]]],
) -> list[list[str]]:
    keys = make_history_key_frame(query_frame)
    result = []

    for query, location, category, delivery, params in zip(
        keys["__query"],
        keys["__location"],
        keys["__category"],
        keys["__delivery"],
        keys["__params"],
    ):
        context_key = (
            query,
            location,
            category,
            delivery,
            params,
        )
        query_location_key = (query, location)
        query_key = (query,)

        selected = []
        seen = set()

        for source in (
            history_maps["context"].get(context_key, ()),
            history_maps["query_location"].get(query_location_key, ()),
            history_maps["query"].get(query_key, ()),
        ):
            for item_id in source:
                if item_id not in seen:
                    seen.add(item_id)
                    selected.append(item_id)

        result.append(selected)

    return result


In [ ]:
def inject_history_candidates(
    baseline_rankings: list[np.ndarray],
    history_candidates: list[list[str]],
    item_id_to_index: dict[str, int],
    history_k: int,
    final_k: int = 50,
) -> list[np.ndarray]:
    result = []

    for baseline, historical_item_ids in zip(
        baseline_rankings,
        history_candidates,
    ):
        selected = []
        seen = set()

        for item_id in historical_item_ids[:history_k]:
            item_index = item_id_to_index.get(str(item_id))
            if item_index is not None and item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)

        for raw_item_index in baseline:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)
            if len(selected) == final_k:
                break

        assert len(selected) == final_k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


In [ ]:
HISTORY_K = 10

item_id_set = set(items["item_id"].astype(str))
history_maps = build_history_maps(
    history_events=train,
    allowed_item_ids=item_id_set,
)
history_candidates = get_history_candidates(
    query_frame=queries,
    history_maps=history_maps,
)
item_id_to_index = {
    str(item_id): index
    for index, item_id in enumerate(lexical_index["item_ids"])
}
history_top50 = inject_history_candidates(
    baseline_rankings=lexical_top50,
    history_candidates=history_candidates,
    item_id_to_index=item_id_to_index,
    history_k=HISTORY_K,
    final_k=50,
)
print(
    "Запросов с историческими кандидатами: "
    f"{np.mean([len(row) > 0 for row in history_candidates]):.2%}"
)


## 3. Кандидаты по предсказанной микрокатегории

Линейный классификатор обучается на текстах запросов и параметрах.
Для каждого benchmark-запроса берётся наиболее вероятная микрокатегория,
после чего из глубокой BM25F-выдачи добавляются 20 объявлений этого класса.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import FeatureUnion


def build_query_intent_texts(frame: pd.DataFrame) -> list[str]:
    queries_text = frame["search_query"].fillna("").astype(str)
    params_text = frame["search_infm_params_text"].fillna("").astype(str)
    return (queries_text + " params " + params_text).tolist()


def make_microcategory_vectorizer() -> FeatureUnion:
    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        preprocessor=lemmatize_text,
        tokenizer=str.split,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.997,
        max_features=250_000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        preprocessor=normalize_lexical_text,
        lowercase=False,
        ngram_range=(3, 5),
        min_df=3,
        max_features=350_000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    return FeatureUnion(
        [
            ("word", word_vectorizer),
            ("char", char_vectorizer),
        ],
        transformer_weights={
            "word": 1.0,
            "char": 0.8,
        },
        n_jobs=1,
    )


def make_microcategory_classifier() -> SGDClassifier:
    return SGDClassifier(
        loss="log_loss",
        penalty="l2",
        alpha=2e-6,
        max_iter=30,
        tol=1e-4,
        average=True,
        random_state=SEED,
        n_jobs=-1,
    )


def top_class_labels(
    classifier: SGDClassifier,
    feature_matrix,
    max_k: int = 10,
) -> np.ndarray:
    scores = classifier.decision_function(feature_matrix)

    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])

    max_k = min(max_k, scores.shape[1])
    top_positions = np.argsort(
        -scores,
        axis=1,
        kind="stable",
    )[:, :max_k]
    return classifier.classes_[top_positions]


In [ ]:
def build_microcategory_expert_candidates(
    retrieval: dict,
    item_microcategories: np.ndarray,
    predicted_microcategories: np.ndarray,
    predicted_top_k: int,
    strategy: str = "sequential",
) -> list[np.ndarray]:
    if strategy not in {"sequential", "round_robin"}:
        raise ValueError(f"Неизвестная strategy={strategy}")

    result = []

    for query_position, predicted in enumerate(predicted_microcategories):
        local_ranking = retrieval["location_indices"][query_position]
        global_ranking = retrieval["global_indices"][query_position]
        per_category = []

        for microcategory in predicted[:predicted_top_k]:
            category_items = []
            category_seen = set()

            for ranking in (local_ranking, global_ranking):
                matching = ranking[
                    item_microcategories[ranking] == microcategory
                ]

                for raw_item_index in matching:
                    item_index = int(raw_item_index)
                    if item_index not in category_seen:
                        category_seen.add(item_index)
                        category_items.append(item_index)

            per_category.append(category_items)

        if strategy == "sequential":
            selected = [
                item_index
                for category_items in per_category
                for item_index in category_items
            ]
        else:
            # По одному кандидату каждого класса за цикл. Благодаря этому
            # predicted_top_k=3/5 действительно использует не только top-1.
            selected = []
            max_category_size = max(map(len, per_category), default=0)
            for depth in range(max_category_size):
                for category_items in per_category:
                    if depth < len(category_items):
                        selected.append(category_items[depth])

        result.append(np.asarray(selected, dtype=np.int32))

    return result


def inject_index_candidates(
    baseline_rankings: list[np.ndarray],
    expert_candidates: list[np.ndarray],
    expert_k: int,
    final_k: int = 50,
) -> list[np.ndarray]:
    result = []

    for baseline, expert in zip(baseline_rankings, expert_candidates):
        selected = []
        seen = set()

        for raw_item_index in expert[:expert_k]:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)

        for raw_item_index in baseline:
            item_index = int(raw_item_index)
            if item_index not in seen:
                seen.add(item_index)
                selected.append(item_index)
            if len(selected) == final_k:
                break

        assert len(selected) == final_k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


In [ ]:
MICROCAT_PREDICTED_K = 1
MICROCAT_CANDIDATE_K = 20

microcat_train_mask = (
    train["item_microcat_id"].notna()
    & train["search_query"].notna()
)
microcat_train = train.loc[microcat_train_mask]
train_intent_texts = build_query_intent_texts(microcat_train)
benchmark_intent_texts = build_query_intent_texts(queries)

microcat_vectorizer = make_microcategory_vectorizer()
start = time.perf_counter()
microcat_train_matrix = microcat_vectorizer.fit_transform(train_intent_texts)
microcat_query_matrix = microcat_vectorizer.transform(benchmark_intent_texts)
print(f"Векторизация микрокатегорий: {time.perf_counter() - start:.1f} с")

microcat_targets = normalize_identifier(microcat_train["item_microcat_id"])
microcat_classifier = make_microcategory_classifier()
start = time.perf_counter()
microcat_classifier.fit(microcat_train_matrix, microcat_targets)
print(f"Обучение классификатора: {time.perf_counter() - start:.1f} с")

predicted_microcategories = top_class_labels(
    classifier=microcat_classifier,
    feature_matrix=microcat_query_matrix,
    max_k=MICROCAT_PREDICTED_K,
)
item_microcategories = normalize_identifier(items["item_microcat_id"])
microcat_candidates = build_microcategory_expert_candidates(
    retrieval=lexical_retrieval,
    item_microcategories=item_microcategories,
    predicted_microcategories=predicted_microcategories,
    predicted_top_k=MICROCAT_PREDICTED_K,
    strategy="sequential",
)
microcat_top50 = inject_index_candidates(
    baseline_rankings=history_top50,
    expert_candidates=microcat_candidates,
    expert_k=MICROCAT_CANDIDATE_K,
    final_k=50,
)

del microcat_train_matrix, microcat_query_matrix
del microcat_vectorizer, microcat_classifier
del train_intent_texts, benchmark_intent_texts
gc.collect()


## 4. Векторный поиск E5

Запрос кодируется вместе с параметрами, объявление — вместе с заголовком,
параметрами и описанием. Из объединённой локальной и глобальной выдачи добавляются
первые 20 кандидатов.


In [ ]:
DENSE_MODEL_NAME = "intfloat/multilingual-e5-small"
# Текущая ревизия модели зафиксирована, чтобы обновления репозитория
# Hugging Face не меняли результат.
DENSE_MODEL_REVISION = "fd1525a9fd15316a2d503bf26ab031a61d056e98"
DENSE_MAX_LENGTH = 160
DENSE_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DENSE_BATCH_SIZE = 256 if DENSE_DEVICE == "cuda" else 64
DENSE_LOCATION_QUOTA = 20
DENSE_CANDIDATE_K = 20


def clean_dense_field(values: pd.Series) -> pd.Series:
    return (
        values.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def build_dense_query_texts(frame: pd.DataFrame) -> list[str]:
    query = clean_dense_field(frame["search_query"])
    params = clean_dense_field(frame["search_infm_params_text"])
    return ("query: " + query + ". " + params).tolist()


def build_dense_item_texts(frame: pd.DataFrame) -> list[str]:
    title = clean_dense_field(frame["item_title_raw"])
    params = clean_dense_field(frame["item_infm_params_text"])
    description = clean_dense_field(frame["item_description_raw"])
    return ("passage: " + title + ". " + params + ". " + description).tolist()


def encode_dense_texts(model, texts: list[str]) -> np.ndarray:
    embeddings = model.encode(
        texts,
        batch_size=DENSE_BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return np.ascontiguousarray(embeddings, dtype=np.float32)


In [ ]:
def exact_dense_global_search(
    item_embeddings: np.ndarray,
    query_embeddings: np.ndarray,
    item_ids: np.ndarray,
    k: int = 100,
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    # Порядок корпуса не зависит от разметки валидации.
    corpus_order = np.argsort(item_ids, kind="stable")
    ordered_embeddings = np.ascontiguousarray(
        item_embeddings[corpus_order],
        dtype=np.float32,
    )

    index = faiss.IndexFlatIP(item_embeddings.shape[1])
    index.add(ordered_embeddings)
    scores, positions = index.search(query_embeddings, k)

    rankings = [
        corpus_order[row].astype(np.int32, copy=False)
        for row in positions
    ]
    return rankings, [row.astype(np.float32, copy=False) for row in scores]


def exact_dense_location_search(
    item_embeddings: np.ndarray,
    query_embeddings: np.ndarray,
    item_ids: np.ndarray,
    item_locations: np.ndarray,
    query_locations: np.ndarray,
    k: int = 100,
) -> tuple[list[np.ndarray], list[np.ndarray]]:
    rankings = [None] * len(query_embeddings)
    score_rows = [None] * len(query_embeddings)

    for location in pd.unique(query_locations):
        query_positions = np.flatnonzero(query_locations == location)
        item_positions = np.flatnonzero(item_locations == location)

        if len(item_positions) == 0:
            for query_position in query_positions:
                rankings[query_position] = np.empty(0, dtype=np.int32)
                score_rows[query_position] = np.empty(0, dtype=np.float32)
            continue

        location_order = np.argsort(
            item_ids[item_positions],
            kind="stable",
        )
        ordered_items = item_positions[location_order]

        index = faiss.IndexFlatIP(item_embeddings.shape[1])
        index.add(
            np.ascontiguousarray(
                item_embeddings[ordered_items],
                dtype=np.float32,
            )
        )

        local_k = min(k, len(ordered_items))
        scores, positions = index.search(
            np.ascontiguousarray(
                query_embeddings[query_positions],
                dtype=np.float32,
            ),
            local_k,
        )

        for offset, query_position in enumerate(query_positions):
            rankings[query_position] = ordered_items[
                positions[offset]
            ].astype(np.int32, copy=False)
            score_rows[query_position] = scores[offset].astype(
                np.float32,
                copy=False,
            )

    return rankings, score_rows


def fuse_dense_local_global(
    local_rankings: list[np.ndarray],
    global_rankings: list[np.ndarray],
    local_quota: int,
    k: int = 50,
) -> list[np.ndarray]:
    result = []

    for local_ranking, global_ranking in zip(
        local_rankings,
        global_rankings,
    ):
        selected = []
        seen = set()

        for source, limit in (
            (local_ranking, local_quota),
            (global_ranking, k),
        ):
            source_added = 0
            for raw_item_index in source:
                item_index = int(raw_item_index)
                if item_index in seen:
                    continue
                seen.add(item_index)
                selected.append(item_index)
                source_added += 1
                if source_added == limit or len(selected) == k:
                    break
            if len(selected) == k:
                break

        assert len(selected) == k
        result.append(np.asarray(selected, dtype=np.int32))

    return result


In [ ]:
dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    revision=DENSE_MODEL_REVISION,
    device=DENSE_DEVICE,
)
dense_model.max_seq_length = DENSE_MAX_LENGTH
print(f"Устройство E5: {DENSE_DEVICE}")

start = time.perf_counter()
dense_item_embeddings = encode_dense_texts(
    dense_model,
    build_dense_item_texts(items),
)
dense_query_embeddings = encode_dense_texts(
    dense_model,
    build_dense_query_texts(queries),
)
print(f"Кодирование E5: {time.perf_counter() - start:.1f} с")

dense_global, _ = exact_dense_global_search(
    item_embeddings=dense_item_embeddings,
    query_embeddings=dense_query_embeddings,
    item_ids=lexical_index["item_ids"],
    k=100,
)
dense_local, _ = exact_dense_location_search(
    item_embeddings=dense_item_embeddings,
    query_embeddings=dense_query_embeddings,
    item_ids=lexical_index["item_ids"],
    item_locations=lexical_index["item_locations"],
    query_locations=query_locations,
    k=100,
)
dense_rankings = fuse_dense_local_global(
    local_rankings=dense_local,
    global_rankings=dense_global,
    local_quota=DENSE_LOCATION_QUOTA,
    k=50,
)
dense_top50 = inject_index_candidates(
    baseline_rankings=microcat_top50,
    expert_candidates=dense_rankings,
    expert_k=DENSE_CANDIDATE_K,
    final_k=50,
)


## 5. Перенос локации по истории

Если в корпусе нет объявлений с точным `search_location_id`, по train
выбирается наиболее частая локация объявления для этой локации поиска.
Правило применяется только при достаточной поддержке и доле перехода.


In [ ]:
LEARNED_LOCATION_MIN_SUPPORT = 50
LEARNED_LOCATION_MIN_SHARE = 0.60
NO_LOCATION_EXPERT = "<NO_LOCATION_EXPERT>"


def predict_learned_item_locations(
    history_events: pd.DataFrame,
    query_frame: pd.DataFrame,
    item_frame: pd.DataFrame,
    min_support: int = LEARNED_LOCATION_MIN_SUPPORT,
    min_share: float = LEARNED_LOCATION_MIN_SHARE,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    history = pd.DataFrame({
        "search_location": normalize_identifier(
            history_events["search_location_id"]
        ),
        "item_location": normalize_identifier(
            history_events["item_location_id"]
        ),
    })
    history = history.loc[
        history["search_location"].ne("<NA>")
        & history["item_location"].ne("<NA>")
    ]

    transition_counts = (
        history
        .groupby(["search_location", "item_location"], sort=False)
        .size()
        .rename("count")
        .reset_index()
    )
    transition_counts["support"] = (
        transition_counts
        .groupby("search_location")["count"]
        .transform("sum")
    )
    dominant = (
        transition_counts
        .sort_values(
            ["search_location", "count", "item_location"],
            ascending=[True, False, True],
            kind="mergesort",
        )
        .drop_duplicates("search_location")
        .copy()
    )
    dominant["dominant_share"] = dominant["count"] / dominant["support"]
    dominant_by_search = dominant.set_index("search_location").to_dict("index")

    available_item_locations = set(
        normalize_identifier(item_frame["item_location_id"])
    )
    query_locations = normalize_identifier(query_frame["search_location_id"])

    predictions = []
    eligible = []
    report_rows = []

    for query_position, search_location in enumerate(query_locations):
        stats = dominant_by_search.get(search_location)
        exact_location_available = search_location in available_item_locations

        if stats is None:
            predicted_location = NO_LOCATION_EXPERT
            support = 0
            dominant_share = 0.0
        else:
            predicted_location = str(stats["item_location"])
            support = int(stats["support"])
            dominant_share = float(stats["dominant_share"])

        use_expert = (
            not exact_location_available
            and predicted_location in available_item_locations
            and predicted_location != search_location
            and support >= min_support
            and dominant_share >= min_share
        )

        predictions.append(
            predicted_location if use_expert else NO_LOCATION_EXPERT
        )
        eligible.append(use_expert)
        report_rows.append({
            "query_position": query_position,
            "search_location": search_location,
            "predicted_item_location": predicted_location,
            "exact_location_available": exact_location_available,
            "transition_support": support,
            "dominant_share": dominant_share,
            "eligible": use_expert,
        })

    return (
        np.asarray(predictions, dtype=object),
        np.asarray(eligible, dtype=bool),
        pd.DataFrame(report_rows),
    )


In [ ]:
LEARNED_LOCATION_K = 10

learned_locations, learned_location_mask, learned_location_report = (
    predict_learned_item_locations(
        history_events=train,
        query_frame=queries,
        item_frame=items,
    )
)
learned_location_rankings, _ = exact_dense_location_search(
    item_embeddings=dense_item_embeddings,
    query_embeddings=dense_query_embeddings,
    item_ids=lexical_index["item_ids"],
    item_locations=lexical_index["item_locations"],
    query_locations=learned_locations,
    k=LEARNED_LOCATION_K,
)
learned_location_candidates = [
    ranking if use_rule else np.empty(0, dtype=np.int32)
    for ranking, use_rule in zip(learned_location_rankings, learned_location_mask)
]
final_top50 = inject_index_candidates(
    baseline_rankings=dense_top50,
    expert_candidates=learned_location_candidates,
    expert_k=LEARNED_LOCATION_K,
    final_k=50,
)
print(f"Перенос локации применён к {learned_location_mask.sum():,} запросам")


## 6. Сохранение ответа

Перед записью проверяются все ограничения формата. Переменную
`EXPECTED_ANSWER_SHA256` можно заполнить хешем отправленного файла;
тогда несовпадение остановит выполнение.


In [ ]:
EXPECTED_ANSWER_SHA256 = os.environ.get("AVITO_EXPECTED_SHA256", "").strip()


def build_answer(
    query_ids: pd.Series,
    rankings: list[np.ndarray],
    item_ids: np.ndarray,
) -> pd.DataFrame:
    rows = []
    for ranking in rankings:
        selected = item_ids[ranking[:50]].astype(str).tolist()
        assert len(selected) == 50
        assert len(set(selected)) == 50
        rows.append(" ".join(selected))
    return pd.DataFrame({
        "query_id": query_ids.astype(str).to_numpy(),
        "answer": rows,
    })


answer = build_answer(
    query_ids=queries["query_id"],
    rankings=final_top50,
    item_ids=lexical_index["item_ids"],
)
valid_item_ids = set(items["item_id"].astype(str))

assert answer.columns.tolist() == ["query_id", "answer"]
assert len(answer) == len(queries)
assert answer["query_id"].is_unique
assert set(answer["query_id"]) == set(queries["query_id"].astype(str))
assert answer["query_id"].str.fullmatch(r".{16}").all()

for prediction in answer["answer"]:
    predicted_ids = prediction.split()
    assert len(predicted_ids) == 50
    assert len(set(predicted_ids)) == 50
    assert all(re.fullmatch(r"[0-9a-f]{16}", item_id) for item_id in predicted_ids)
    assert set(predicted_ids) <= valid_item_ids

answer_path = OUTPUT_DIR / "answer.csv"
answer.to_csv(
    answer_path,
    index=False,
    encoding="utf-8",
    lineterminator="\n",
)
answer_sha256 = hashlib.sha256(answer_path.read_bytes()).hexdigest()

if EXPECTED_ANSWER_SHA256:
    assert answer_sha256 == EXPECTED_ANSWER_SHA256, (
        f"SHA-256 не совпал: {answer_sha256} != {EXPECTED_ANSWER_SHA256}"
    )

print(f"Сохранено: {answer_path}")
print(f"Строк: {len(answer):,}")
print(f"SHA-256: {answer_sha256}")
display(answer.head())
